# WMH Benchmark — U-Net vs MedSAM-LoRA

## Purpose

This notebook performs a strict evaluation-only benchmark between:

- U-Net baseline
- MedSAM ViT-B with the final custom LoRA configuration

No retraining, fine-tuning, optimizer update, or checkpoint modification is performed.

The benchmark uses the completed reproduction run:

- MedSAM-LoRA checkpoint:
  `reproduction_runs/run_001/checkpoints/best_lora_wmh.pth`

- U-Net checkpoint:
  `unet_baseline/checkpoints/best_unet_wmh.pth`

---

## Evaluation cohort

Both models are evaluated on the identical held-out test cohort:

- Patients: 9
- Sites: Amsterdam, Singapore, Utrecht
- Axial slices: 537
- Positive slices: 232
- Empty slices: 305

The original patient ordering and slice correspondence are preserved.

Both models receive identical target masks and are evaluated on the same scoring grid.

---

## Benchmark protocol

The benchmark reports:

1. Overall per-slice metrics
2. Site-wise performance
3. Positive-slice and empty-slice behavior
4. Paired per-slice results for statistical analysis

Metrics:

- Dice coefficient
- IoU
- Sensitivity
- Precision

All results are generated from the saved checkpoints and existing dataset manifests.

---

## Important scientific notes

### MedSAM prompt setting

MedSAM-LoRA evaluation uses the original reference-box prompt protocol from the reproduction workflow.

This represents a box-prompted segmentation setting rather than a fully automatic WMH segmentation pipeline.

### U-Net comparison

The previous U-Net experiment used a different evaluation setup.  
This notebook rebuilds a fair comparison by evaluating both models on the same 537-slice cohort, including empty slices.

The U-Net checkpoint is loaded only for inference; no retraining is performed.

### Metric interpretation

Historical reproduction metrics and this unified benchmark are reported separately.

The benchmark numbers should not be interpreted as direct replacements for the original reproduction headline metrics because aggregation protocols differ.

---

## Reproducibility

This notebook records:

- checkpoint hashes
- dataset hashes
- cohort verification
- model configuration
- generated CSV outputs

The generated benchmark files are stored separately from the reproduction run.

## 1. Environment setup


### Code cell 01 — Paths and fixed benchmark protocol


In [1]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/WMH_MedSAM_Project')
REPRO_RUN_DIR = PROJECT_DIR / 'reproduction_runs/run_001'
TEST_CSV = PROJECT_DIR / 'manifests/test.csv'
WMH_ROOT = PROJECT_DIR / 'data/wmh_dataset/wmh_data'
MEDSAM_CHECKPOINT = REPRO_RUN_DIR / 'checkpoints/best_lora_wmh.pth'
UNET_CHECKPOINT = PROJECT_DIR / 'unet_baseline/checkpoints/best_unet_wmh.pth'
MEDSAM_SOURCE_DIR = Path('/content/MedSAM')
BENCHMARK_DIR = PROJECT_DIR / 'benchmark_runs/unet_vs_medsam_002'

# Use the original completed reproduction run by default, not the older
# project-level checkpoint with the same filename. No automatic fallback.
MEDSAM_RUN_LABEL = 'reproduction_run_001_epoch_9'
UNET_RUN_LABEL = 'research_unet_epoch_20'
EXPECTED_MEDSAM_EPOCH = 9
EXPECTED_UNET_EPOCH = 20
EXPECTED_MEDSAM_VAL_DICE = 0.8891934836701392
EXPECTED_UNET_VAL_DICE = 0.660066099551439
EXPECTED_MEDSAM_SHA256 = None  # set to a verified checkpoint hash when available
EXPECTED_UNET_SHA256 = None
RECORDED_BASE_SHA256 = '34b34b78c1d18cb8c6bf84cf9c00e135d6d6c965699f3c0e31ef1bc9dcb5be74'

REPRO_ENV_PATH = REPRO_RUN_DIR / 'configs/environment.json'
REPRO_INPUT_HASHES_PATH = REPRO_RUN_DIR / 'configs/input_sha256.json'
REPRO_TEST_COPY = REPRO_RUN_DIR / 'configs/original_test.csv'
REQUIREMENTS_LOCK = None  # optionally supply the same environment lock
INSTALL_LOCK = False
MEDSAM_COMMIT = None  # recovered from REPRO_ENV_PATH when available

SEED = 42
BATCH_SIZE = 2
NUM_WORKERS = 4
SCORE_SIZE = (1024, 1024)
THRESHOLD = 0.5
MODELS = ['MedSAM-LoRA', 'U-Net']
KEYS = ['site', 'patient_id', 'slice_id']
METRICS = ['dice', 'iou', 'sensitivity', 'precision']
EXPECTED_TEST_ORDER = [
    ('Singapore', '60'), ('Amsterdam', '112'), ('Utrecht', '49'),
    ('Utrecht', '23'), ('Amsterdam', '137'), ('Singapore', '66'),
    ('Singapore', '65'), ('Utrecht', '21'), ('Amsterdam', '116'),
]
EXPECTED_SITE_SLICES = {'Amsterdam': 249, 'Singapore': 144, 'Utrecht': 144}
assert BATCH_SIZE == 2 and SCORE_SIZE == (1024, 1024) and THRESHOLD == 0.5


### Code cell 02 — Mount Drive and use the recorded source/environment


In [2]:
import importlib.util
import json
import re
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    print('Local Jupyter: configure paths in cell 01.')
else:
    drive.mount('/content/drive')

repro_environment = json.loads(REPRO_ENV_PATH.read_text()) if REPRO_ENV_PATH.is_file() else {}
if INSTALL_LOCK:
    if REQUIREMENTS_LOCK is None or not Path(REQUIREMENTS_LOCK).is_file():
        raise FileNotFoundError('Supply the original requirements lock before INSTALL_LOCK=True.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS_LOCK)], check=True)
if MEDSAM_COMMIT is None:
    MEDSAM_COMMIT = repro_environment.get('medsam_commit')
if MEDSAM_COMMIT is not None and not re.fullmatch(r'[0-9a-fA-F]{40}', MEDSAM_COMMIT):
    raise ValueError('Use a full verified MedSAM commit hash.')
if not MEDSAM_SOURCE_DIR.exists():
    if MEDSAM_COMMIT is None:
        raise FileNotFoundError('Provide the same MedSAM source checkout or its recorded commit; do not guess a revision.')
    subprocess.run(['git', 'clone', 'https://github.com/bowang-lab/MedSAM.git',
                    str(MEDSAM_SOURCE_DIR)], check=True)
    subprocess.run(['git', '-C', str(MEDSAM_SOURCE_DIR), 'checkout', '--detach', MEDSAM_COMMIT], check=True)
if not (MEDSAM_SOURCE_DIR / 'segment_anything/__init__.py').is_file():
    raise FileNotFoundError('Invalid MedSAM source directory.')
missing = [name for name in ['torch', 'torchvision', 'numpy', 'pandas', 'nibabel', 'cv2', 'tqdm']
           if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f'Missing dependencies {missing}; use the original configured environment.')
if 'segment_anything' in sys.modules:
    raise RuntimeError('Restart the kernel so the configured source is imported unambiguously.')
sys.path.insert(0, str(MEDSAM_SOURCE_DIR.resolve()))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Code cell 03 — Imports, provenance helpers and fresh output directory


In [3]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import gc
import hashlib
import random
import platform
import numpy as np
import pandas as pd
import nibabel as nib
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display
import segment_anything
from segment_anything import sam_model_registry

if not torch.cuda.is_available():
    raise RuntimeError('Use CUDA for the original AMP inference paths.')
device = torch.device('cuda', torch.cuda.current_device())
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
if not Path(segment_anything.__file__).resolve().is_relative_to(MEDSAM_SOURCE_DIR.resolve()):
    raise RuntimeError('Wrong segment_anything source imported.')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def array_sha256(array):
    a = np.ascontiguousarray(array)
    return hashlib.sha256(str((a.shape, a.dtype.str)).encode() + a.tobytes()).hexdigest()

for path in [TEST_CSV, MEDSAM_CHECKPOINT, UNET_CHECKPOINT]:
    if not path.is_file():
        raise FileNotFoundError(f'Required existing input: {path}. No retraining fallback is provided.')
if BENCHMARK_DIR.exists() and any(BENCHMARK_DIR.iterdir()):
    raise FileExistsError('Choose a new empty BENCHMARK_DIR to avoid stale predictions/results.')
CONFIG_DIR = BENCHMARK_DIR / 'configs'
PRED_DIR = BENCHMARK_DIR / 'logits'
for folder in [CONFIG_DIR, PRED_DIR / 'medsam', PRED_DIR / 'unet']:
    folder.mkdir(parents=True, exist_ok=True)
input_checkpoint_hashes = {'MedSAM-LoRA': sha256_file(MEDSAM_CHECKPOINT), 'U-Net': sha256_file(UNET_CHECKPOINT)}
for model, expected in [('MedSAM-LoRA', EXPECTED_MEDSAM_SHA256), ('U-Net', EXPECTED_UNET_SHA256)]:
    if expected is not None:
        assert input_checkpoint_hashes[model] == expected, model
source_hashes = {str(p.relative_to(MEDSAM_SOURCE_DIR)): sha256_file(p)
                 for p in sorted((MEDSAM_SOURCE_DIR / 'segment_anything').rglob('*.py'))}
if repro_environment.get('medsam_source_sha256'):
    assert source_hashes == repro_environment['medsam_source_sha256'], 'MedSAM source differs from reproduction record'
else:
    print('Original source fingerprints unavailable; current source will be recorded, not claimed verified.')
if repro_environment.get('cuda_matmul_allow_tf32') is not None:
    torch.backends.cuda.matmul.allow_tf32 = repro_environment['cuda_matmul_allow_tf32']
if repro_environment.get('cudnn_allow_tf32') is not None:
    torch.backends.cudnn.allow_tf32 = repro_environment['cudnn_allow_tf32']
print('GPU:', torch.cuda.get_device_name(device), 'torch:', torch.__version__)


GPU: Tesla T4 torch: 2.11.0+cu128


## 2. Load the exact test split


### Code cell 04 — Verify original CSV order and resolve the same patient files


In [4]:
test_csv_hash = sha256_file(TEST_CSV)
if REPRO_TEST_COPY.is_file():
    assert test_csv_hash == sha256_file(REPRO_TEST_COPY), 'Not the same test CSV as the reproduction run'
recorded_test_hash = repro_environment.get('split_sha256', {}).get('test')
if recorded_test_hash:
    assert test_csv_hash == recorded_test_hash
test_df = pd.read_csv(TEST_CSV)
assert {'site', 'patient_id', 'image_path', 'mask_path'}.issubset(test_df.columns)
assert list(zip(test_df['site'], test_df['patient_id'].astype(str))) == EXPECTED_TEST_ORDER
assert not test_df.duplicated(['site', 'patient_id']).any()
assert test_df['site'].value_counts().to_dict() == dict.fromkeys(EXPECTED_SITE_SLICES, 3)
(CONFIG_DIR / 'original_test.csv').write_bytes(TEST_CSV.read_bytes())

resolved = []
for row in test_df.itertuples(index=False):
    candidates = [p for p in (WMH_ROOT / 'training' / row.site).rglob('wmh.nii')
                  if p.parent.name == str(row.patient_id)]
    if len(candidates) != 1:
        raise ValueError(f'Expected one mask for {row.site}/{row.patient_id}; got {len(candidates)}')
    mask_path = candidates[0]
    image_path = mask_path.parent / 'pre/FLAIR.nii'
    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    resolved.append((str(image_path), str(mask_path)))
test_df = test_df.copy()
test_df['image_path'] = [x[0] for x in resolved]
test_df['mask_path'] = [x[1] for x in resolved]
test_df.to_csv(CONFIG_DIR / 'test_resolved.csv', index=False)
display(test_df[['site', 'patient_id']])


,site,patient_id
0,Singapore,60
1,Amsterdam,112
2,Utrecht,49
3,Utrecht,23
4,Amsterdam,137
5,Singapore,66
6,Singapore,65
7,Utrecht,21
8,Amsterdam,116


### Code cell 05 — Copy the finalized MedSAM dataset unchanged


In [5]:
class WMHMedSAMDataset(Dataset):

    def __init__(self, df):
        self.samples = []
        for _, row in tqdm(df.iterrows(), total=len(df)):
            flair = nib.load(row['image_path']).get_fdata()
            for z in range(flair.shape[2]):
                self.samples.append({'image_path': row['image_path'], 'mask_path': row['mask_path'], 'slice_id': z, 'patient_id': row['patient_id'], 'site': row['site']})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        flair = nib.load(item['image_path']).get_fdata()
        mask = nib.load(item['mask_path']).get_fdata()
        z = item['slice_id']
        image = flair[:, :, z]
        mask = mask[:, :, z]
        image = (image - image.min()) / (image.max() - image.min() + 1e-08)
        image = np.stack([image, image, image], axis=-1)
        image = cv2.resize(image, (1024, 1024), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask.astype(np.float32), (1024, 1024), interpolation=cv2.INTER_NEAREST)
        ys, xs = np.where(mask > 0)
        if len(xs) > 0:
            box = np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=np.float32)
        else:
            box = np.array([0, 0, 0, 0], dtype=np.float32)
        image = torch.tensor(image.transpose(2, 0, 1), dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.float32)
        box = torch.tensor(box, dtype=torch.float32)
        return {'image': image, 'mask': mask, 'box': box, 'patient_id': item['patient_id'], 'site': item['site'], 'slice_id': item['slice_id']}


### Code cell 06 — Canonical 537-slice index, label audit and loader


In [6]:
test_dataset = WMHMedSAMDataset(test_df)
assert len(test_dataset) == 537
canonical_rows = []
data_hashes, data_audit = {}, []
for row in test_df.itertuples(index=False):
    image = nib.load(row.image_path).get_fdata()
    mask = nib.load(row.mask_path).get_fdata()
    assert image.ndim == 3 and image.shape == mask.shape
    assert np.isfinite(image).all() and np.isfinite(mask).all()
    data_audit.append({'site': row.site, 'patient_id': str(row.patient_id),
                       'shape': list(image.shape), 'labels': np.unique(mask).tolist()})
    for kind, path in [('image', row.image_path), ('mask', row.mask_path)]:
        data_hashes[f'{row.site}/{row.patient_id}/{kind}'] = sha256_file(path)
    for z in range(image.shape[2]):
        target = cv2.resize(mask[:, :, z].astype(np.float32), SCORE_SIZE,
                            interpolation=cv2.INTER_NEAREST)
        target_binary = (target > 0).astype(np.uint8)
        canonical_rows.append({'sample_index': len(canonical_rows), 'site': row.site,
            'patient_id': str(row.patient_id), 'slice_id': z,
            'gt_pixels': int(target_binary.sum()), 'is_positive': bool(target_binary.any()),
            'target_sha256': array_sha256(target_binary)})
cohort_df = pd.DataFrame(canonical_rows)
expected_keys = [tuple(row) for row in cohort_df[KEYS].itertuples(index=False, name=None)]
dataset_keys = [(s['site'], str(s['patient_id']), int(s['slice_id'])) for s in test_dataset.samples]
assert dataset_keys == expected_keys and len(set(expected_keys)) == 537
assert cohort_df['site'].value_counts().to_dict() == EXPECTED_SITE_SLICES
assert (int(cohort_df['is_positive'].sum()), int((~cohort_df['is_positive']).sum())) == (232, 305)
if REPRO_INPUT_HASHES_PATH.is_file():
    original_hashes = json.loads(REPRO_INPUT_HASHES_PATH.read_text())
    assert all(original_hashes.get(key) == value for key, value in data_hashes.items())
else:
    print('Original data hashes unavailable; exact test identities/order checked, current bytes recorded.')
(CONFIG_DIR / 'test_data_hashes.json').write_text(json.dumps(data_hashes, indent=2))
(CONFIG_DIR / 'test_label_audit.json').write_text(json.dumps(data_audit, indent=2))
cohort_df.to_csv(CONFIG_DIR / 'test_slice_index.csv', index=False)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=4,
                         pin_memory=True, drop_last=False)
assert len(test_loader) == 269
sample = test_dataset[0]
assert sample['image'].shape == (3, 1024, 1024) and sample['mask'].shape == (1024, 1024)
assert sample['box'].shape == (4,)
display(cohort_df.groupby('site', sort=False).agg(slices=('slice_id', 'size'), positive=('is_positive', 'sum')))
print('Paired cohort: 537 slices, 232 positive, 305 empty; no filtering.')
del image, mask, sample


  0%|          | 0/9 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


,slices,positive
site,,
Singapore,144,80
Amsterdam,249,71
Utrecht,144,81


Paired cohort: 537 slices, 232 positive, 305 empty; no filtering.


## 3. Load MedSAM-LoRA checkpoint


### Code cell 07 — Exact LoRA class, targets and MedSAM forward


In [7]:
class LoRALinear(nn.Module):

    def __init__(self, original_layer, rank=4):
        super().__init__()
        self.original = original_layer
        for p in self.original.parameters():
            p.requires_grad = False
        self.lora_A = nn.Linear(original_layer.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, original_layer.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=5 ** 0.5)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.original(x) + self.lora_B(self.lora_A(x))

def add_lora_controlled(model):
    count = 0
    targets = ['image_encoder.blocks', 'mask_decoder.transformer.layers', 'mask_decoder.transformer.final_attn_token_to_image']
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear):
            if any((t in name for t in targets)):
                parent = model
                parts = name.split('.')
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                setattr(parent, parts[-1], LoRALinear(module))
                count += 1
    return count

def forward_logits(model, image, box):
    image_embeddings = model.image_encoder(image)
    sparse_embeddings, dense_embeddings = model.prompt_encoder(points=None, boxes=box, masks=None)
    low_res_masks, _ = model.mask_decoder(image_embeddings=image_embeddings, image_pe=model.prompt_encoder.get_dense_pe(), sparse_prompt_embeddings=sparse_embeddings, dense_prompt_embeddings=dense_embeddings, multimask_output=False)
    assert image_embeddings.shape == (image.shape[0], 256, 64, 64)
    assert low_res_masks.shape == (image.shape[0], 1, 256, 256)
    pred_mask = F.interpolate(low_res_masks, size=(1024, 1024), mode='bilinear', align_corners=False)
    return pred_mask.squeeze(1)


### Code cell 08 — Construct ViT-B and strictly load the entire saved model


In [11]:
checkpoint_metadata = {}

BASE_MEDSAM_CHECKPOINT = PROJECT_DIR / 'checkpoints/medsam_vit_b.pth'

if not BASE_MEDSAM_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f'Base MedSAM checkpoint not found: {BASE_MEDSAM_CHECKPOINT}'
    )


def load_full_checkpoint(path, expected_epoch, expected_dice):
    state = torch.load(path, map_location='cpu', weights_only=True)

    if not isinstance(state, dict) or not isinstance(state.get('model_state_dict'), dict):
        raise ValueError(
            'Expected a full saved checkpoint with model_state_dict; '
            'no adapter-only or partial-load fallback.'
        )

    epoch = int(state['epoch'])
    dice = float(state['dice'])

    assert epoch == expected_epoch, (str(path), epoch, expected_epoch)
    assert abs(dice - expected_dice) < 1e-12, (
        str(path), dice, expected_dice
    )

    return state


# Build original MedSAM ViT-B from base checkpoint
sam = sam_model_registry['vit_b'](
    checkpoint=str(BASE_MEDSAM_CHECKPOINT)
)

# Inject the exact Final LoRA configuration
assert add_lora_controlled(sam) == 80

for name, p in sam.named_parameters():
    p.requires_grad = ('lora_' in name)


assert sum(
    p.numel() for p in sam.parameters() if p.requires_grad
) == 673792

assert sum(
    p.numel() for p in sam.parameters()
) == 94409264


adapters = [
    (name, module)
    for name, module in sam.named_modules()
    if isinstance(module, LoRALinear)
]


assert len(adapters) == 80

assert [
    sum(name.startswith(prefix) for name, _ in adapters)
    for prefix in [
        'image_encoder.blocks.',
        'mask_decoder.transformer.layers.',
        'mask_decoder.transformer.final_attn_token_to_image.'
    ]
] == [48, 28, 4]


assert all(
    m.lora_A.out_features == m.lora_B.in_features == 4
    for _, m in adapters
)

assert all(
    not hasattr(m, 'alpha') and not hasattr(m, 'scaling')
    for _, m in adapters
)


# Load final LoRA checkpoint
state = load_full_checkpoint(
    MEDSAM_CHECKPOINT,
    EXPECTED_MEDSAM_EPOCH,
    EXPECTED_MEDSAM_VAL_DICE
)


if state.get('split_sha256', {}).get('test'):
    assert state['split_sha256']['test'] == test_csv_hash


if state.get('base_checkpoint_sha256'):
    assert (
        state['base_checkpoint_sha256']
        == RECORDED_BASE_SHA256
    )


sam.load_state_dict(
    state['model_state_dict'],
    strict=True
)


checkpoint_metadata['MedSAM-LoRA'] = {
    'path': str(MEDSAM_CHECKPOINT),
    'sha256': input_checkpoint_hashes['MedSAM-LoRA'],
    'run_label': MEDSAM_RUN_LABEL,
    'epoch': int(state['epoch']),
    'validation_dice': float(state['dice']),
    'base_sha256': state.get('base_checkpoint_sha256'),
    'adapter_parameters': 673792,
    'total_parameters': 94409264,
    'lora_layers': 80,
    'rank': 4,
}


del state, adapters

sam.to(device).eval()

sam.requires_grad_(False)

assert not sam.training
assert not any(
    p.requires_grad for p in sam.parameters()
)


print(checkpoint_metadata['MedSAM-LoRA'])

{'path': '/content/drive/MyDrive/WMH_MedSAM_Project/reproduction_runs/run_001/checkpoints/best_lora_wmh.pth', 'sha256': '803156a14542733964fc49927255d3cf523c50ad8b829fbd3db1d96b114290e4', 'run_label': 'reproduction_run_001_epoch_9', 'epoch': 9, 'validation_dice': 0.8891934836701392, 'base_sha256': '34b34b78c1d18cb8c6bf84cf9c00e135d6d6c965699f3c0e31ef1bc9dcb5be74', 'adapter_parameters': 673792, 'total_parameters': 94409264, 'lora_layers': 80, 'rank': 4}


## 4. Run MedSAM-LoRA inference


### Code cell 09 — Cache exact MedSAM logits with canonical slice keys


In [12]:
def save_logits(model_name, folder, index, key, logits):
    array = logits.cpu().numpy()
    assert array.ndim == 2 and np.isfinite(array).all()
    path = folder / f'{index:04d}_{key[0]}_{key[1]}_{key[2]:03d}.npz'
    if path.exists():
        raise FileExistsError('Cached prediction already exists; use a fresh benchmark directory.')
    np.savez_compressed(path, logits=array)
    return {'sample_index': index, 'site': key[0], 'patient_id': key[1], 'slice_id': key[2],
            'model': model_name, 'path': str(path), 'sha256': sha256_file(path),
            'height': array.shape[0], 'width': array.shape[1], 'dtype': str(array.dtype)}

medsam_cache_rows = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='MedSAM-LoRA inference'):
        with torch.amp.autocast('cuda'):
            logits = forward_logits(sam, batch['image'].to(device), batch['box'].to(device))
        assert logits.shape == (len(batch['site']), 1024, 1024)
        for i, site in enumerate(batch['site']):
            key = (site, str(int(batch['patient_id'][i])), int(batch['slice_id'][i]))
            index = len(medsam_cache_rows)
            assert key == expected_keys[index]
            medsam_cache_rows.append(save_logits('MedSAM-LoRA', PRED_DIR / 'medsam', index, key, logits[i]))
assert len(medsam_cache_rows) == 537
pd.DataFrame(medsam_cache_rows).to_csv(CONFIG_DIR / 'medsam_cache_index.csv', index=False)
del sam, logits, batch
gc.collect()
torch.cuda.empty_cache()
print('Saved 537 MedSAM outputs; released MedSAM GPU memory.')


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


MedSAM-LoRA inference:   0%|          | 0/269 [00:00<?, ?it/s]

Saved 537 MedSAM outputs; released MedSAM GPU memory.


## 5. Load U-Net architecture and aligned evaluation dataset


### Code cell 10 — Exact research U-Net architecture


In [13]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.down1 = DoubleConv(1, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.bottom = DoubleConv(512, 1024)
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)
        c2 = self.down2(p1)
        p2 = self.pool(c2)
        c3 = self.down3(p2)
        p3 = self.pool(c3)
        c4 = self.down4(p3)
        p4 = self.pool(c4)
        b = self.bottom(p4)
        u4 = self.up4(b)
        u4 = torch.cat([u4, c4], dim=1)
        u4 = self.conv4(u4)
        u3 = self.up3(u4)
        u3 = torch.cat([u3, c3], dim=1)
        u3 = self.conv3(u3)
        u2 = self.up2(u3)
        u2 = torch.cat([u2, c2], dim=1)
        u2 = self.conv2(u2)
        u1 = self.up1(u2)
        u1 = torch.cat([u1, c1], dim=1)
        u1 = self.conv1(u1)
        return self.out(u1)


### Code cell 11 — All-slice U-Net index; preserve the original input transform


In [14]:
class WMHUNetAllSlicesDataset(Dataset):
    def __init__(self, canonical_samples):
        self.samples = [dict(item) for item in canonical_samples]
        self.size = 256

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        flair = nib.load(item['image_path']).get_fdata()
        mask = nib.load(item['mask_path']).get_fdata()
        z = item['slice_id']
        image = flair[:, :, z]
        mask = mask[:, :, z]
        image = (image - image.min()) / (image.max() - image.min() + 1e-08)
        image = cv2.resize(image, (self.size, self.size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask.astype(np.float32), (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        image = torch.tensor(image[None, :, :], dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.float32)
        return {'image': image, 'mask': mask, 'patient_id': item['patient_id'], 'site': item['site'], 'slice_id': item['slice_id']}

unet_test_dataset = WMHUNetAllSlicesDataset(test_dataset.samples)
unet_keys = [(s['site'], str(s['patient_id']), int(s['slice_id'])) for s in unet_test_dataset.samples]
assert unet_keys == expected_keys and len(unet_test_dataset) == 537
unet_test_loader = DataLoader(unet_test_dataset, batch_size=2, shuffle=False,
                              num_workers=4, pin_memory=True, drop_last=False)
assert len(unet_test_loader) == 269
unet_sample = unet_test_dataset[0]
assert unet_sample['image'].shape == (1, 256, 256)
assert unet_sample['mask'].shape == (256, 256)
print('U-Net: 537 slices, [1,256,256] input; original __getitem__ preserved.')
print('Its 256x256 mask is NOT used as the benchmark scoring target.')
del unet_sample


U-Net: 537 slices, [1,256,256] input; original __getitem__ preserved.
Its 256x256 mask is NOT used as the benchmark scoring target.


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 6. Load U-Net checkpoint


### Code cell 12 — Strictly load existing U-Net weights and BatchNorm buffers


In [15]:
unet = UNet()
state = load_full_checkpoint(UNET_CHECKPOINT, EXPECTED_UNET_EPOCH, EXPECTED_UNET_VAL_DICE)
unet.load_state_dict(state['model_state_dict'], strict=True)
checkpoint_metadata['U-Net'] = {
    'path': str(UNET_CHECKPOINT), 'sha256': input_checkpoint_hashes['U-Net'],
    'run_label': UNET_RUN_LABEL, 'epoch': int(state['epoch']), 'validation_dice': float(state['dice']),
    'parameters': sum(p.numel() for p in unet.parameters()),
    'training_cohort_note': 'Research positive-slice-only training/validation; no retraining here',
}
del state
unet.to(device).eval()
unet.requires_grad_(False)
assert not unet.training and not any(p.requires_grad for p in unet.parameters())
assert all(not m.training for m in unet.modules() if isinstance(m, nn.BatchNorm2d))
print(checkpoint_metadata['U-Net'])


{'path': '/content/drive/MyDrive/WMH_MedSAM_Project/unet_baseline/checkpoints/best_unet_wmh.pth', 'sha256': 'f15b78c13d7e18d93150148a3e3c7509340feb6a67ac3f20ce824da2799c1aba', 'run_label': 'research_unet_epoch_20', 'epoch': 20, 'validation_dice': 0.660066099551439, 'parameters': 31042369, 'training_cohort_note': 'Research positive-slice-only training/validation; no retraining here'}


## 7. Run U-Net inference


### Code cell 13 — Cache original 256x256 U-Net logits for all 537 slices


In [16]:
unet_cache_rows = []
with torch.no_grad():
    for batch in tqdm(unet_test_loader, desc='U-Net inference'):
        with torch.amp.autocast('cuda'):
            logits = unet(batch['image'].to(device)).squeeze(1)
        assert logits.shape == (len(batch['site']), 256, 256)
        for i, site in enumerate(batch['site']):
            key = (site, str(int(batch['patient_id'][i])), int(batch['slice_id'][i]))
            index = len(unet_cache_rows)
            assert key == expected_keys[index]
            unet_cache_rows.append(save_logits('U-Net', PRED_DIR / 'unet', index, key, logits[i]))
assert len(unet_cache_rows) == 537
pd.DataFrame(unet_cache_rows).to_csv(CONFIG_DIR / 'unet_cache_index.csv', index=False)
del unet, logits, batch
gc.collect()
torch.cuda.empty_cache()
print('Saved 537 U-Net outputs; no empty slice was skipped.')


U-Net inference:   0%|          | 0/269 [00:00<?, ?it/s]

Saved 537 U-Net outputs; no empty slice was skipped.


## 8. Unified metric calculation


### Code cell 14 — One shared binary metric function and one grid-alignment rule


In [17]:
def calculate_metrics_final(pred, target, threshold=0.5):
    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()
    target = (target > 0).float()
    pred = pred.view(-1)
    target = target.view(-1)
    tp = (pred * target).sum().item()
    fp = (pred * (1 - target)).sum().item()
    fn = ((1 - pred) * target).sum().item()
    dice = (2 * tp + 1e-07) / (2 * tp + fp + fn + 1e-07)
    iou = (tp + 1e-07) / (tp + fp + fn + 1e-07)
    sensitivity = (tp + 1e-07) / (tp + fn + 1e-07)
    precision = (tp + 1e-07) / (tp + fp + 1e-07)
    return {'dice': min(dice, 1.0), 'iou': min(iou, 1.0), 'sensitivity': min(sensitivity, 1.0), 'precision': min(precision, 1.0)}

def align_logits_for_scoring(logits):
    assert logits.ndim == 3 and logits.shape[0] == 1
    if tuple(logits.shape[-2:]) != SCORE_SIZE:
        logits = F.interpolate(logits.unsqueeze(1), size=SCORE_SIZE,
                               mode='bilinear', align_corners=False).squeeze(1)
    assert tuple(logits.shape) == (1, 1024, 1024)
    return logits

def load_cached_logits(record):
    assert sha256_file(record['path']) == record['sha256'], 'Prediction cache changed'
    with np.load(record['path'], allow_pickle=False) as content:
        array = content['logits']
    assert array.shape == (record['height'], record['width'])
    assert str(array.dtype) == record['dtype'] and np.isfinite(array).all()
    return torch.from_numpy(array).unsqueeze(0).to(device)


### Code cell 15 — Check empty-mask conventions and scoring-grid behavior


In [18]:
with torch.no_grad():
    zeros = torch.zeros((1, 2, 2), device=device)
    neg = torch.full_like(zeros, -20)
    pos = torch.full_like(zeros, 20)
    assert calculate_metrics_final(neg, zeros) == dict.fromkeys(METRICS, 1.0)
    assert calculate_metrics_final(pos, torch.ones_like(zeros)) == dict.fromkeys(METRICS, 1.0)
    assert calculate_metrics_final(pos, zeros)['dice'] < 1e-6
    assert calculate_metrics_final(neg, torch.ones_like(zeros))['dice'] < 1e-6
    assert calculate_metrics_final(pos, zeros)['sensitivity'] == 1.0
    assert calculate_metrics_final(neg, torch.ones_like(zeros))['precision'] == 1.0
    assert calculate_metrics_final(zeros, torch.ones_like(zeros))['dice'] < 1e-6
    assert calculate_metrics_final(pos, 2 * torch.ones_like(zeros)) == dict.fromkeys(METRICS, 1.0)
    assert align_logits_for_scoring(torch.ones((1, 256, 256), device=device)).shape == (1, 1024, 1024)
print('Shared metric edge cases and output-grid alignment verified.')


Shared metric edge cases and output-grid alignment verified.


### Code cell 16 — Score both models against the very same 1024x1024 target


In [19]:
metric_rows = []
with torch.no_grad():
    for index in tqdm(range(537), desc='Unified per-slice scoring'):
        sample = test_dataset[index]
        key = (sample['site'], str(sample['patient_id']), int(sample['slice_id']))
        assert key == expected_keys[index]
        target = sample['mask'].unsqueeze(0).to(device)
        target_bin = (target > 0).float()
        target_hash = array_sha256((sample['mask'].numpy() > 0).astype(np.uint8))
        assert target_hash == canonical_rows[index]['target_sha256']
        for record in [medsam_cache_rows[index], unet_cache_rows[index]]:
            assert tuple(record[k] for k in KEYS) == key
            with torch.amp.autocast('cuda'):
                logits = align_logits_for_scoring(load_cached_logits(record))
            values = calculate_metrics_final(logits, target, threshold=THRESHOLD)
            pred_bin = (torch.sigmoid(logits) > THRESHOLD).float()
            tp = int((pred_bin * target_bin).sum().item())
            fp = int((pred_bin * (1 - target_bin)).sum().item())
            fn = int(((1 - pred_bin) * target_bin).sum().item())
            assert tp + fn == canonical_rows[index]['gt_pixels']
            assert all(np.isfinite(v) and 0 <= v <= 1 for v in values.values())
            metric_rows.append({
                'sample_index': index, 'site': key[0], 'patient_id': key[1], 'slice_id': key[2],
                'model': record['model'], 'is_positive': canonical_rows[index]['is_positive'],
                'gt_pixels': tp + fn, 'pred_pixels': tp + fp, 'tp': tp, 'fp': fp, 'fn': fn,
                'target_sha256': target_hash, 'score_height': 1024, 'score_width': 1024,
                **values,
            })
per_slice = pd.DataFrame(metric_rows)
assert len(per_slice) == 1074
assert not per_slice.duplicated(['model'] + KEYS).any()
for model in MODELS:
    rows = per_slice[per_slice['model'] == model]
    assert list(rows[KEYS].itertuples(index=False, name=None)) == expected_keys
    assert rows['site'].value_counts().to_dict() == EXPECTED_SITE_SLICES
assert per_slice.groupby(KEYS)['target_sha256'].nunique().eq(1).all()
print('Verified 537 paired keys and identical target hashes for both models.')


Unified per-slice scoring:   0%|          | 0/537 [00:00<?, ?it/s]

Verified 537 paired keys and identical target hashes for both models.


## 9. Site-wise comparison and documented aggregation


### Code cell 17 — Compute matched overall, site, subgroup and patient summaries


In [20]:
def summarize_slice_metrics(frame, groups):
    rows = []
    group_arg = groups[0] if len(groups) == 1 else groups
    for key, part in frame.groupby(group_arg, sort=False):
        values = (key,) if len(groups) == 1 else key
        rows.append({**dict(zip(groups, values)), 'slices': len(part),
            'patients': len(part[['site', 'patient_id']].drop_duplicates()),
            'positive_slices': int(part['is_positive'].sum()),
            'empty_slices': int((~part['is_positive']).sum()),
            **{metric: sum(part[metric].tolist()) / len(part) for metric in METRICS}})
    return pd.DataFrame(rows)

overall = summarize_slice_metrics(per_slice, ['model'])
sitewise = summarize_slice_metrics(per_slice, ['model', 'site'])
patientwise = summarize_slice_metrics(per_slice, ['model', 'site', 'patient_id'])
subgroup_frame = per_slice.assign(subgroup=np.where(per_slice['is_positive'], 'positive', 'empty'))
subgroups = summarize_slice_metrics(subgroup_frame, ['model', 'subgroup'])

left = per_slice[per_slice['model'] == MODELS[0]]
right = per_slice[per_slice['model'] == MODELS[1]]
paired = left[KEYS + METRICS].merge(right[KEYS + METRICS], on=KEYS,
        how='outer', validate='one_to_one', indicator=True, suffixes=('_medsam', '_unet'))
assert len(paired) == 537 and paired['_merge'].eq('both').all()
paired = cohort_df[['sample_index'] + KEYS + ['is_positive']].merge(
    paired.drop(columns='_merge'), on=KEYS, validate='one_to_one', sort=False)
for metric in METRICS:
    paired[f'{metric}_difference_medsam_minus_unet'] = paired[f'{metric}_medsam'] - paired[f'{metric}_unet']
assert len(overall) == 2 and len(sitewise) == 6 and len(patientwise) == 18
assert overall['slices'].eq(537).all()
assert overall['positive_slices'].eq(232).all() and overall['empty_slices'].eq(305).all()

# Primary overall is the slice-weighted combination of site means, not an
# unweighted mean of three sites and not the historical mean of batch scores.
for model in MODELS:
    parts = sitewise[sitewise['model'] == model]
    row = overall[overall['model'] == model].iloc[0]
    for metric in METRICS:
        assert np.isclose(sum(parts[metric] * parts['slices']) / 537, row[metric], atol=1e-12, rtol=0)
display(overall)
display(sitewise)
display(subgroups)
print('Overall/site metrics are means of per-slice binary scores on the common grid.')


,model,slices,patients,positive_slices,empty_slices,dice,iou,sensitivity,precision
0,MedSAM-LoRA,537,9,232,305,0.877692,0.817739,0.898520,0.884887
1,U-Net,537,9,232,305,0.668270,0.612527,0.863615,0.699936


,model,site,slices,patients,positive_slices,empty_slices,dice,iou,sensitivity,precision
0,MedSAM-LoRA,Singapore,144,3,80,64,0.847664,0.777564,0.841714,0.901946
1,U-Net,Singapore,144,3,80,64,0.638987,0.575668,0.789289,0.736474
2,MedSAM-LoRA,Amsterdam,249,3,71,178,0.921991,0.880330,0.938855,0.916336
3,U-Net,Amsterdam,249,3,71,178,0.664951,0.624195,0.939463,0.654346
4,MedSAM-LoRA,Utrecht,144,3,81,63,0.831122,0.749685,0.885581,0.813448
5,U-Net,Utrecht,144,3,81,63,0.703291,0.629210,0.806787,0.742231


,model,subgroup,slices,patients,positive_slices,empty_slices,dice,iou,sensitivity,precision
0,MedSAM-LoRA,empty,305,9,0,305,1.000000,1.000000,1.000000,1.000000
1,U-Net,empty,305,9,0,305,0.711475,0.711475,1.000000,0.711475
2,MedSAM-LoRA,positive,232,9,232,0,0.716900,0.578129,0.765109,0.733554
3,U-Net,positive,232,9,232,0,0.611470,0.482444,0.684315,0.684766


Overall/site metrics are means of per-slice binary scores on the common grid.


## 10. Save CSV results and provenance


### Code cell 18 — Write new benchmark outputs only; verify input checkpoints unchanged


In [21]:
csv_outputs = {
    'benchmark_overall.csv': overall,
    'benchmark_sitewise.csv': sitewise,
    'benchmark_per_slice.csv': per_slice,
    'benchmark_paired_slices.csv': paired,
    'benchmark_positive_empty.csv': subgroups,
    'benchmark_patient_slice_means.csv': patientwise,
}
for filename, frame in csv_outputs.items():
    frame.to_csv(BENCHMARK_DIR / filename, index=False)
for model, path in [('MedSAM-LoRA', MEDSAM_CHECKPOINT), ('U-Net', UNET_CHECKPOINT)]:
    assert sha256_file(path) == input_checkpoint_hashes[model], 'Input checkpoint changed during benchmark'
assert sha256_file(TEST_CSV) == test_csv_hash
for row in test_df.itertuples(index=False):
    for kind, path in [('image', row.image_path), ('mask', row.mask_path)]:
        assert sha256_file(path) == data_hashes[f'{row.site}/{row.patient_id}/{kind}']

freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(CONFIG_DIR / 'requirements_observed.txt').write_text(freeze)
provenance = {
    'protocol': 'paired_537_slices_1024_grid_binary_target_slice_macro_mean',
    'training_performed': False, 'test_csv': str(TEST_CSV), 'test_csv_sha256': test_csv_hash,
    'ordered_test_patients': EXPECTED_TEST_ORDER, 'site_slices': EXPECTED_SITE_SLICES,
    'checkpoints': checkpoint_metadata, 'data_sha256': data_hashes,
    'medsam_source_sha256': source_hashes, 'recorded_medsam_commit': MEDSAM_COMMIT,
    'python': sys.version, 'platform': platform.platform(), 'torch': str(torch.__version__),
    'cuda_build': torch.version.cuda, 'gpu': torch.cuda.get_device_name(device),
    'cudnn': torch.backends.cudnn.version(), 'seed': SEED,
    'cuda_autocast': True, 'batch_size': BATCH_SIZE, 'workers': NUM_WORKERS,
    'strict_deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
    'cuda_matmul_allow_tf32': torch.backends.cuda.matmul.allow_tf32,
    'cudnn_allow_tf32': torch.backends.cudnn.allow_tf32,
    'mask_rule': 'target > 0; values 1 and 2 both foreground',
    'score_size': SCORE_SIZE, 'threshold': THRESHOLD,
    'unet_output_alignment': 'bilinear logits to 1024x1024, align_corners=False, before sigmoid',
    'medsam_prompt': 'ground-truth box; zero box for empty masks',
    'unet_prompt': 'none',
    'primary_aggregation': 'arithmetic mean of per-slice metrics over 537 slices',
    'patient_report': 'mean of patient slice metrics, not volumetric Dice',
    'reproduction_file_modified': False,
    'csv_sha256': {filename: sha256_file(BENCHMARK_DIR / filename) for filename in csv_outputs},
}
(BENCHMARK_DIR / 'benchmark_provenance.json').write_text(json.dumps(provenance, indent=2))
for filename in csv_outputs:
    print(BENCHMARK_DIR / filename)
print('Input checkpoint/data hashes unchanged. New benchmark outputs:', BENCHMARK_DIR)


/content/drive/MyDrive/WMH_MedSAM_Project/benchmark_runs/unet_vs_medsam_002/benchmark_overall.csv
/content/drive/MyDrive/WMH_MedSAM_Project/benchmark_runs/unet_vs_medsam_002/benchmark_sitewise.csv
/content/drive/MyDrive/WMH_MedSAM_Project/benchmark_runs/unet_vs_medsam_002/benchmark_per_slice.csv
/content/drive/MyDrive/WMH_MedSAM_Project/benchmark_runs/unet_vs_medsam_002/benchmark_paired_slices.csv
/content/drive/MyDrive/WMH_MedSAM_Project/benchmark_runs/unet_vs_medsam_002/benchmark_positive_empty.csv
/content/drive/MyDrive/WMH_MedSAM_Project/benchmark_runs/unet_vs_medsam_002/benchmark_patient_slice_means.csv
Input checkpoint/data hashes unchanged. New benchmark outputs: /content/drive/MyDrive/WMH_MedSAM_Project/benchmark_runs/unet_vs_medsam_002


## Completion checks and interpretation

| Verification | Required result |
|---|---|
| Exact patient sequence | CSV sequence printed in P code 05 saved output |
| Test slices / paired keys | 537 / 537 |
| Positive / empty slices for each model | 232 / 305 |
| Site slices | Amsterdam 249; Singapore 144; Utrecht 144 |
| MedSAM inputs / masks / boxes | [B,3,1024,1024] / [B,1024,1024] / [B,4] |
| U-Net input / original output | [B,1,256,256] / [B,1,256,256] |
| Scored outputs / shared target | [1,1024,1024] for both models |
| MedSAM adapters / rank / adapter parameters | 80 / 4 / 673,792 |
| MedSAM total parameters | 94,409,264 |
| Parameters requiring gradients during inference | 0 for both models |
| Per-slice long table / paired table | 1,074 / 537 rows |
| Overall / site / patient-summary rows | 2 / 6 / 18 |
| Checkpoint load | strict=True; no missing/unexpected keys |
| Checkpoint and data file hashes after evaluation | unchanged |

The notebook contains all definitions in execution order and does not import or
run the research/reproduction notebooks. The class and function excerpts are
embedded so neither original notebook must be edited. Cached predictions are
indexed and checked by `(site, patient_id, slice_id)`; missing, duplicate, stale,
or mismatched predictions stop scoring rather than being silently skipped.

Do not compare the new unified overall Dice directly with historical 0.8739 or
0.6852 as evidence of a model improvement/decline: those values used different
aggregation/target/cohort conventions. Interpret the two new rows together under
this documented protocol. The benchmark measures these two saved systems under
an aligned scoring setup; it does not isolate architecture, prompt access,
training-distribution, or resolution effects.


## Source notebook fingerprints

- `WMH_MedSAM_LoRA_Research_Log(1).ipynb`: SHA256 `3ad1771c10d0c63e7cc6afcb02479a46af27cc4a845d46d387edbd2ce801dabe`.
- `WMH_MedSAM_LoRA_Reproduction (2)(1).ipynb`: SHA256 `43c8bf31c4734ef4909c57e071ecd32b268aa9b4e03867f420f4b46bdb8297db`.
